# EX_10 — LangGraph y flujos (ejercicios)

**Notebook de referencia:** `notebook/10_LangGraph_Flujos.ipynb`

**Tiempo orientativo:** ~30 minutos.


## Actividad 1 — Estado TypedDict

Define un `TypedDict` de estado con al menos: `question: str`, `answer: str`, `step_count: int`.


In [1]:
from typing import TypedDict

class QAState(TypedDict):
    question: str
    answer: str
    step_count: int

# Estado inicial de ejemplo
initial_state: QAState = {
    "question": "¿Qué es un agente de IA?",
    "answer": "",
    "step_count": 0,
}
print(initial_state)


{'question': '¿Qué es un agente de IA?', 'answer': '', 'step_count': 0}


## Actividad 2 — Dos nodos

Esquematiza (pseudocódigo) nodos `retrieve` y `generate` que incrementen `step_count`. No hace falta ejecutar LangGraph si aún no está importado en el entorno.


In [3]:
# Pseudocódigo de nodos LangGraph (patrón del notebook 10)

def retrieve(state: QAState) -> QAState:
    """Simula recuperación de contexto relevante para la pregunta."""
    # docs = retriever.invoke(state["question"])
    # context = "\n".join(d.page_content for d in docs)
    context = f"[contexto simulado para: {state['question']}]"
    return {
        **state,
        "step_count": state["step_count"] + 1,
        # En un grafo real añadirías: "context": context
    }

def generate(state: QAState) -> QAState:
    """Simula generación de respuesta a partir del contexto recuperado."""
    # answer = llm.invoke(prompt_with_context).content
    answer = f"Respuesta simulada a: {state['question']}"
    return {
        **state,
        "answer": answer,
        "step_count": state["step_count"] + 1,
    }

# Demostración del flujo secuencial retrieve → generate
s0 = {"question": "¿Qué es un agente de IA?", "answer": "", "step_count": 0}
s1 = retrieve(s0)
s2 = generate(s1)
print("Tras retrieve:", s1)
print("Tras generate:", s2)


Tras retrieve: {'question': '¿Qué es un agente de IA?', 'answer': '', 'step_count': 1}
Tras generate: {'question': '¿Qué es un agente de IA?', 'answer': 'Respuesta simulada a: ¿Qué es un agente de IA?', 'step_count': 2}


## Actividad 3 — Condicional

Describe en markdown cuándo enrutarías a un nodo `human_review` (p. ej. si `confidence < 0.5`).


**Criterio de enrutamiento hacia `human_review`:**

Enrutaría al nodo `human_review` cuando la **confianza de la respuesta generada sea baja** (`confidence < 0.5`), por ejemplo si:

| Señal | Umbral / condición |
|-------|-------------------|
| Score del reranker | Ningún chunk supera 0.5 de relevancia |
| Verificador LLM | El nodo `check_quality` devuelve `MALA` o `MEJORABLE` tras 2 iteraciones |
| Pregunta sensible | Clasificador detecta categoría `APPEAL`, `LEGAL` o `OTHER` con baja certeza |
| Respuesta vacía | `answer == ""` o contiene *"no tengo información"* en consultas de alto impacto |

**Función de routing (pseudocódigo):**

```python
def route_after_generate(state) -> Literal["finalize", "human_review"]:
    if state.get("confidence", 1.0) < 0.5:
        return "human_review"
    if state.get("response_quality") == "MALA":
        return "human_review"
    return "finalize"
```

Así el grafo no entrega automáticamente respuestas dudosas en dominios críticos (impuestos, salud, legal).
